In [1]:
from pathlib import Path
from typing import Iterable, Sequence

import dtale
import pandas as pd

In [2]:
# Function definitions
Filter = dict[str, str | Iterable[str]]


def pprint_sermon(data: pd.Series) -> str:
    """Generate string pretty-printing sermon represented by series data.

    Works with both ekklesia and subsplash data.
    """
    sermon_id = int(data["sermon_id"]) if not pd.isna(data["sermon_id"]) else 0
    title = data["title"] if not pd.isna(data["title"]) else "untitled"
    speaker = data["speaker"] if not pd.isna(data["speaker"]) else "unknown speaker"
    if not pd.isna(data["audio"]):
        media = "audio"
    elif not pd.isna(data["video"]):
        media = "video"
    else:
        media = "unknown media"
    return f"{sermon_id:>8}: {title} ({speaker}) ({media})"


def print_key_comparison(
    key: str,
    additional_keys_to_display: Sequence[str] = (),
    print_values: bool = False,
    df_ekklesia: pd.DataFrame | None = None,
    df_subsplash: pd.DataFrame | None = None,
) -> None:
    """Print comparison of uniqueness of key between Ekklesia and Subsplash.

    :param key: data column key to analyze
    :param print_values: if flag raised, renders display of individual values. If not, only prints
    comparison summary values.
    :param additional_keys_to_display: sequence of columns other than key to print if print_values
    is raised. Only uses if all printed entries are unique.
    :param df_ekklesia: Ekklesia dataset to use. If not given, uses the notebook's default
        ekklesia data.
    :param df_subsplash: Subsplash dataset to use. If not given, uses the notebook's default
        subsplash data.
    """
    df_ekklesia = df_ekklesia if df_ekklesia is not None else data_ekklesia
    df_subsplash = df_subsplash if df_subsplash is not None else data_subsplash
    ekklesia_keys = set(df_ekklesia[key].dropna().unique())
    subsplash_keys = set(df_subsplash[key].dropna().unique())
    subsplash_only_keys = subsplash_keys - ekklesia_keys
    ekklesia_only_keys = ekklesia_keys - subsplash_keys
    print(f"--------- Ekklesia {key} ----------------------------------")
    print(f" Entries w/o: {len(df_ekklesia) - len(ekklesia_keys)}")
    print(f"       Total: {len(ekklesia_keys)}")
    print(f"Total unique: {len(ekklesia_only_keys)}")
    print(f"--------- Subsplash {key} ----------------------------------")
    print(f" Entries w/o: {len(df_subsplash) - len(subsplash_keys)}")
    print(f"       Total: {len(subsplash_keys)}")
    print(f"Total unique: {len(subsplash_only_keys)}")

    if print_values:
        df = pd.DataFrame({key: sorted(subsplash_keys | ekklesia_keys)})
        df["in_ekklesia"] = df[key].isin(ekklesia_keys)
        df["in_subsplash"] = df[key].isin(subsplash_keys)
        print(dtale.show(df))

In [ ]:
# Load data files
data_ekklesia = pd.read_csv(Path("../data/ekklesia/sermonExportSep_22_2026.csv"))
data_subsplash = pd.read_csv(Path("../data/20260822_subsplash_sermonexport.csv"))

data_subsplash.columns = data_subsplash.columns.str.lower()
data_subsplash.rename(
    inplace=True,
    columns={
        "sermon_title": "title",
    },
)
data_subsplash["sermon_id"] = data_subsplash["sermon_id"].astype("Int64")
data_subsplash["audio_filename"] = data_subsplash["audio"].str.split("/").str[-1]
data_subsplash["date"] = pd.to_datetime(
    data_subsplash["date"], format="mixed", errors="coerce", utc=True
)
data_subsplash["source"] = "subsplash"

data_ekklesia.columns = data_ekklesia.columns.str.lower()
data_ekklesia.rename(
    inplace=True,
    columns={
        "preachers": "speaker",
        "sermon id": "sermon_id",
    },
)
data_ekklesia["audio_filename"] = data_ekklesia["audio"].str.split("/").str[-1]
data_ekklesia["date"] = pd.to_datetime(
    data_ekklesia["date"], format="mixed", errors="coerce", utc=True
)
data_ekklesia["source"] = "ekklesia"

# Sort by date
data_ekklesia.sort_values(by="date")
data_subsplash.sort_values(by="date")

print(
    f" Ekklesia Keys: {','.join(sorted(data_ekklesia.keys()))}\n"
    f"Subsplash Keys: {','.join(sorted(data_subsplash.keys()))}"
)

In [ ]:
"""Question: How many entries are missing audio?

Answer: one entry in Ekklesia is missing audio, and looks kinda like an empty line (only a sermon_id)
five different entries in Subsplash are missing audio. Maybe most of them are videos? two of them are unknown, though.

"""
print("--------- Ekklesia Entries w/o audio ----------------------------------")
for _, series in data_ekklesia.iloc[data_ekklesia["audio"].isna()].iterrows():
    print(pprint_sermon(series))
print("--------- Subsplash Entries w/o audio ---------------------------------")
for _, series in data_subsplash.iloc[data_subsplash["audio"].isna()].iterrows():
    print(pprint_sermon(series))

In [ ]:
# print_key_comparison("sermon_id", print_values=True)
print_key_comparison("speaker", print_values=True)

In [ ]:
target_series = "1 Corinthians"
print_key_comparison(
    "audio",
    print_values=True,
    df_subsplash=data_subsplash[data_subsplash["series"] == target_series],
    df_ekklesia=data_ekklesia[data_ekklesia["series"] == target_series],
)

In [50]:
# Question: Some audio file entries that we can find are the same file, but hosted in different locations. For example:
print(dtale.show(pd.DataFrame([data_ekklesia.iloc[592], data_subsplash.iloc[839]])))